In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/opinionx'
print("Setup done")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done


In [4]:
df = pd.read_csv(f'{BASE}/data/sample/sample_50k.csv')

print(f"Loaded: {len(df):,} reviews")
print(f"Columns: {list(df.columns)}")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())

Loaded: 40,379 reviews
Columns: ['star_rating', 'helpful_votes', 'total_votes', 'vine', 'verified_purchase', 'review_headline', 'review_body', 'product_title', 'product_category', 'sentiment', 'label', 'review_length', 'helpfulness_ratio', 'clean_text', 'clean_headline', 'full_text']

Sentiment distribution:
sentiment
positive    25000
negative    15379
Name: count, dtype: int64


In [5]:
# Upsample minority class (negative) to match positive
df_positive = df[df['sentiment'] == 'positive']
df_negative = df[df['sentiment'] == 'negative']

df_negative_upsampled = resample(
    df_negative,
    replace=True,
    n_samples=len(df_positive),
    random_state=42
)

df_balanced = pd.concat([df_positive, df_negative_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced dataset: {len(df_balanced):,} reviews")
print(df_balanced['sentiment'].value_counts())

Balanced dataset: 50,000 reviews
sentiment
negative    25000
positive    25000
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X = df_balanced['clean_text'].fillna('')
y = df_balanced['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {len(X_train):,}")
print(f"Test size:  {len(X_test):,}")

Train size: 40,000
Test size:  10,000


In [7]:
tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),      # unigrams + bigrams
    min_df=2,                 # ignore very rare terms
    max_df=0.95,              # ignore very common terms
    sublinear_tf=True         # apply log normalization
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")
print(f"Train matrix shape: {X_train_tfidf.shape}")

Vocabulary size: 50,000
Train matrix shape: (40000, 50000)


In [8]:
def evaluate_model(name, y_test, y_pred):
    print(f"\n{'='*45}")
    print(f"Model: {name}")
    print(f"{'='*45}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=['Negative', 'Positive']))
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1': round(f1_score(y_test, y_pred), 4)
    }

results = []
print("Evaluation function ready")

Evaluation function ready


In [9]:
print("Training Naive Bayes...")
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
results.append(evaluate_model("Naive Bayes", y_test, y_pred_nb))

Training Naive Bayes...

Model: Naive Bayes
Accuracy:  0.9362
Precision: 0.9364
Recall:    0.9360
F1 Score:  0.9362

Classification Report:
              precision    recall  f1-score   support

    Negative       0.94      0.94      0.94      5000
    Positive       0.94      0.94      0.94      5000

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000



In [10]:
print("Training Logistic Regression...")
lr = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    random_state=42
)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
results.append(evaluate_model("Logistic Regression", y_test, y_pred_lr))

Training Logistic Regression...

Model: Logistic Regression
Accuracy:  0.9411
Precision: 0.9485
Recall:    0.9328
F1 Score:  0.9406

Classification Report:
              precision    recall  f1-score   support

    Negative       0.93      0.95      0.94      5000
    Positive       0.95      0.93      0.94      5000

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000



In [11]:
print("Training Linear SVM...")
svm = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
results.append(evaluate_model("Linear SVM", y_test, y_pred_svm))

Training Linear SVM...

Model: Linear SVM
Accuracy:  0.9584
Precision: 0.9662
Recall:    0.9500
F1 Score:  0.9580

Classification Report:
              precision    recall  f1-score   support

    Negative       0.95      0.97      0.96      5000
    Positive       0.97      0.95      0.96      5000

    accuracy                           0.96     10000
   macro avg       0.96      0.96      0.96     10000
weighted avg       0.96      0.96      0.96     10000



In [12]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)

print("\n" + "="*55)
print("MODEL COMPARISON TABLE")
print("="*55)
print(results_df.to_string(index=False))
print("="*55)

# Save results
results_df.to_csv(f'{BASE}/outputs/results.csv', index=False)
print("\nSaved to outputs/results.csv")


MODEL COMPARISON TABLE
              Model  Accuracy  Precision  Recall     F1
         Linear SVM    0.9584     0.9662  0.9500 0.9580
Logistic Regression    0.9411     0.9485  0.9328 0.9406
        Naive Bayes    0.9362     0.9364  0.9360 0.9362

Saved to outputs/results.csv


In [13]:
print("TOP 20 WORDS — Positive Reviews")
print("="*40)
feature_names = tfidf.get_feature_names_out()
lr_coefs = lr.coef_[0]

top_positive_idx = lr_coefs.argsort()[-20:][::-1]
for idx in top_positive_idx:
    print(f"  {feature_names[idx]:<25} {lr_coefs[idx]:.4f}")

print("\nTOP 20 WORDS — Negative Reviews")
print("="*40)
top_negative_idx = lr_coefs.argsort()[:20]
for idx in top_negative_idx:
    print(f"  {feature_names[idx]:<25} {lr_coefs[idx]:.4f}")

TOP 20 WORDS — Positive Reviews
  great                     13.1781
  love                      8.8529
  excellent                 7.7233
  perfect                   7.4810
  good                      6.4538
  works                     6.4282
  perfectly                 6.3916
  nice                      6.1861
  awesome                   5.8180
  amazing                   5.7628
  well                      5.1240
  best                      4.9364
  easy                      4.8534
  little                    4.3698
  price                     4.1134
  easy to                   3.7963
  great product             3.7809
  my                        3.7598
  loves                     3.7096
  not bad                   3.6714

TOP 20 WORDS — Negative Reviews
  not                       -12.6102
  poor                      -6.6961
  disappointed              -5.6894
  broke                     -5.3938
  return                    -5.3029
  terrible                  -5.2295
  returned       

In [14]:
import pickle
import os

models_path = f'{BASE}/outputs/models'
os.makedirs(models_path, exist_ok=True)

# Save all models and vectorizer
pickle.dump(nb, open(f'{models_path}/naive_bayes.pkl', 'wb'))
pickle.dump(lr, open(f'{models_path}/logistic_regression.pkl', 'wb'))
pickle.dump(svm, open(f'{models_path}/linear_svm.pkl', 'wb'))
pickle.dump(tfidf, open(f'{models_path}/tfidf_vectorizer.pkl', 'wb'))

print("All models saved:")
for f in os.listdir(models_path):
    print(f"  {f}")

All models saved:
  distilbert_sentiment
  naive_bayes.pkl
  logistic_regression.pkl
  linear_svm.pkl
  tfidf_vectorizer.pkl
